In [11]:
import os
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.models.segmentation import deeplabv3_resnet50

import matplotlib.pyplot as plt

In [14]:
RAW_DIR   = r"../FishEmbryo_All_Images/FishEmbryo_Raw_Images_Yolk/Yolk"      
LABEL_DIR = r"../FishEmbryo_All_Images/FishEmbryo_Label_Images_BMP/Yolk"    

IMG_SIZE = (512, 512)
BATCH_SIZE = 4
EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-4

USE_SPLIT = True
TEST_RATIO = 0.3

THRESH = 0.5
SEED = 42
NUM_WORKERS = 2

In [15]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [16]:
def list_images(folder):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    folder = Path(folder)
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])

def stem(p: Path):
    return p.stem

def label_to_binary_mask(label_pil: Image.Image) -> np.ndarray:
    """
    Convert label image -> binary mask {0,1}.
    Default rule: any non-zero pixel is yolk.
    If your label uses a specific value/color, adjust this function.
    """
    arr = np.array(label_pil.convert("L"))
    mask = (arr > 0).astype(np.uint8)  # 0/1
    return mask

In [24]:
class YolkDataset(Dataset):
    def __init__(self, raw_dir, label_dir, img_size=(512, 512)):
        raw_paths = list_images(raw_dir)
        label_paths = list_images(label_dir)

        if len(raw_paths) == 0:
            raise RuntimeError(f"No raw images found in: {raw_dir}")
        if len(label_paths) == 0:
            raise RuntimeError(f"No label images found in: {label_dir}")

        label_map = {stem(p): p for p in label_paths}

        self.pairs = []
        missing = 0
        for rp in raw_paths:
            lp = label_map.get(stem(rp))
            if lp is None:
                missing += 1
                continue
            self.pairs.append((rp, lp))

        if len(self.pairs) == 0:
            raise RuntimeError(
                "No raw/label pairs matched by filename stem.\n"
                "Example: ABC_01.bmp (raw) ↔ ABC_01.bmp (label)\n"
                "If label names differ (suffix/prefix), tell me 2 sample names."
            )

        if missing:
            print(f"[WARN] {missing} raw images had no matching label by stem and were skipped.")

        self.img_size = img_size

        # Official torchvision preprocessing for pretrained backbones
        self.img_transform = transforms.Compose([
            transforms.Resize(img_size, interpolation=InterpolationMode.BILINEAR),
            transforms.ToTensor(),  # -> [0,1], [3,H,W]
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]
            ),
        ])

        # Masks: nearest resize to avoid fractional labels
        self.mask_resize = transforms.Resize(img_size, interpolation=InterpolationMode.NEAREST)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        raw_path, label_path = self.pairs[idx]

        # RAW BMP
        img = Image.open(raw_path).convert("RGB")
        x = self.img_transform(img)

        # LABEL BMP
        lbl = Image.open(label_path)
        mask01 = label_to_binary_mask(lbl)           # HxW {0,1}
        mask_pil = Image.fromarray(mask01 * 255)
        mask_pil = self.mask_resize(mask_pil)
        mask = (np.array(mask_pil) > 0).astype(np.float32)  # HxW {0,1}
        y = torch.from_numpy(mask)[None, ...]        # [1,H,W]

        return x, y, str(raw_path)

In [25]:
def build_deeplabv3_binary():
    model = deeplabv3_resnet50(weights="DEFAULT")
    model.classifier[-1] = nn.Conv2d(256, 1, kernel_size=1)  # binary head
    return model

In [26]:
def batch_iou(pred01: torch.Tensor, true01: torch.Tensor, eps=1e-7):
    """
    pred01,true01: [B,1,H,W] in {0,1}
    returns [B] IoU
    """
    pred = pred01.bool()
    true = true01.bool()
    inter = (pred & true).flatten(1).sum(dim=1).float()
    union = (pred | true).flatten(1).sum(dim=1).float()
    return (inter + eps) / (union + eps)


@torch.no_grad()
def eval_mean_iou(model, loader, device, thresh=0.5):
    model.eval()
    ious = []
    for x, y, _ in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)["out"]             # [B,1,H,W]
        probs = torch.sigmoid(logits)
        pred = (probs >= thresh).float()

        ious.append(batch_iou(pred, y).cpu())

    return float(torch.cat(ious).mean().item())

In [27]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss = 0.0
    for x, y, _ in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(x)["out"]
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)["out"]
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)

In [28]:
@torch.no_grad()
def visualize_3(model, dataset, device, thresh=0.5):
    model.eval()
    idxs = [0, len(dataset)//2, len(dataset)-1] if len(dataset) >= 3 else list(range(len(dataset)))

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    for k, idx in enumerate(idxs[:3]):
        x, y, path = dataset[idx]
        x_in = x.unsqueeze(0).to(device)

        logits = model(x_in)["out"][0, 0].cpu()
        prob = torch.sigmoid(logits).numpy()
        pred = (prob >= thresh).astype(np.uint8)

        # de-normalize for display
        x_np = x.cpu().numpy().transpose(1, 2, 0)
        x_np = (x_np * std + mean).clip(0, 1)

        # overlay red
        overlay = x_np.copy()
        alpha = 0.35
        red = np.zeros_like(overlay)
        red[..., 0] = 1.0
        overlay = np.where(pred[..., None] == 1, (1 - alpha) * overlay + alpha * red, overlay)

        plt.figure(figsize=(12, 4))
        plt.suptitle(f"Example {k+1}: {Path(path).name}")

        plt.subplot(1, 3, 1)
        plt.title("Raw")
        plt.imshow(x_np)
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.title("Predicted yolk mask")
        plt.imshow(pred, cmap="gray")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.title("Overlay")
        plt.imshow(overlay)
        plt.axis("off")

        plt.tight_layout()
        plt.show()

In [29]:
def main():
    set_seed(SEED)

    if not os.path.isdir(RAW_DIR):
        raise FileNotFoundError(f"RAW_DIR not found: {RAW_DIR}")
    if not os.path.isdir(LABEL_DIR):
        raise FileNotFoundError(f"LABEL_DIR not found: {LABEL_DIR}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("[INFO] device:", device)

    ds = YolkDataset(RAW_DIR, LABEL_DIR, img_size=IMG_SIZE)
    print("[INFO] paired samples:", len(ds))

    if USE_SPLIT:
        test_len = int(round(len(ds) * TEST_RATIO))
        train_len = len(ds) - test_len
        train_ds, test_ds = random_split(
            ds, [train_len, test_len],
            generator=torch.Generator().manual_seed(SEED)
        )
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True)
        eval_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True)
        print(f"[INFO] split: train={train_len}, test={test_len}")
    else:
        train_loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True)
        eval_loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True)
        print("[INFO] training on ALL images (no split)")

    model = build_deeplabv3_binary().to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    best_iou = -1.0
    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler=scaler)
        miou = eval_mean_iou(model, eval_loader, device, thresh=THRESH)

        if miou > best_iou:
            best_iou = miou
            torch.save(model.state_dict(), "best_deeplabv3_yolk_512.pth")

        print(f"Epoch {epoch:02d}/{EPOCHS} | loss={loss:.4f} | mean IoU={miou:.4f} | best={best_iou:.4f}")

    print(f"\n[INFO] saved: best_deeplabv3_yolk_512.pth")
    print(f"[HW Q4] Mean IoU = {best_iou:.4f}")

    # Q3: show 3 predictions
    model.load_state_dict(torch.load("best_deeplabv3_yolk_512.pth", map_location=device))
    visualize_3(model, ds, device, thresh=THRESH)

In [30]:
if __name__ == "__main__":
    main()

[INFO] device: cuda


RuntimeError: No raw/label pairs matched by filename stem.
Example: ABC_01.bmp (raw) ↔ ABC_01.bmp (label)
If label names differ (suffix/prefix), tell me 2 sample names.